In [2]:
import pandas as pd

# Load Dataset(add path to your dataset)
df = pd.read_csv('IMDB Dataset.csv')

# Prints the first 5 rows
print(df.head())

# Shows number of rows and number of blank rows
print(df.info())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB
None


In [ ]:
# Check for rows where the text is just whitespace or empty
empty_text_rows = df[df['review'].str.strip() == ""]

print(f"Number of perfectly blank emails: {len(empty_text_rows)}")

# If it finds any, this line removes them:
# df = df[df['text'].str.strip() != ""]

# Check how many duplicates exist
print(f"Duplicates: {df.duplicated().sum()}")

# Drop the duplicates and reset the index
df = df.drop_duplicates().reset_index(drop=True)

import re

def clean_text(text):
    # Remove HTML tags like <br />
    text = re.sub(r'<.*?>', ' ', text)
    
    # Remove non-alphanumeric characters (keeps letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Convert to lowercase and strip leading/trailing whitespace
    text = text.lower().strip()
    
    return text

# Apply the function to the review column (assuming your column is named 'review')
df['cleaned_review'] = df['review'].apply(clean_text)

# Let's check out the result
df[['review', 'cleaned_review']].head()

Number of perfectly blank emails: 0
Duplicates: 0


,review,cleaned_review
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming te...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love in the time of money is a ...


In [8]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    df['cleaned_review'], 
    df['sentiment'], 
    test_size=0.2, 
    random_state=42
)

# Initialize the TF-IDF Vectorizer
tfidf = TfidfVectorizer(max_features=5000) # Limits vocabulary to top 5,000 words to save memory

# Fit and transform the training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform the test data 
X_test_tfidf = tfidf.transform(X_test)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Initialize the model
# We set max_iter=1000 just in case it needs a bit more time to converge on a large dataset
model = LogisticRegression(max_iter=1000, random_state=42)

# Train the model using ONLY the training data
print("Training the model...")
model.fit(X_train_tfidf, y_train)

# Make predictions on the unseen test data
print("Making predictions...")
y_pred = model.predict(X_test_tfidf)

# Evaluate the results
accuracy = accuracy_score(y_test, y_pred)
print(f"\nBaseline Accuracy: {accuracy * 100:.2f}%")

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

Training the model...
Making predictions...

Baseline Accuracy: 88.97%

Detailed Classification Report:
              precision    recall  f1-score   support

    negative       0.90      0.87      0.89      4939
    positive       0.88      0.90      0.89      4978

    accuracy                           0.89      9917
   macro avg       0.89      0.89      0.89      9917
weighted avg       0.89      0.89      0.89      9917



In [ ]:
import pandas as pd

# Get the feature names (the words) from the vectorizer
feature_names = tfidf.get_feature_names_out()

# Get the coefficients (the weights) from the model
# model.coef_[0] gives an array of weights corresponding to each feature name
coefficients = model.coef_[0]

# Combine them into a pandas DataFrame 
coef_df = pd.DataFrame({
    'Word': feature_names,
    'Weight': coefficients
})

# Sort to find the top positive and negative words
top_positive = coef_df.sort_values(by='Weight', ascending=False).head(15)
top_negative = coef_df.sort_values(by='Weight', ascending=True).head(15)

print("--- TOP 15 POSITIVE WORDS ---")
print(top_positive.to_string(index=False))

print("\n--- TOP 15 NEGATIVE WORDS ---")
print(top_negative.to_string(index=False))

--- TOP 15 POSITIVE WORDS ---
     Word   Weight
excellent 7.538181
    great 7.471889
  perfect 5.517286
  amazing 5.351309
wonderful 5.046057
     best 5.002529
    loved 4.862598
brilliant 4.679924
hilarious 4.287882
    today 4.265152
  enjoyed 4.195448
   superb 4.114247
 favorite 4.064868
fantastic 3.990401
      fun 3.667169

--- TOP 15 NEGATIVE WORDS ---
         Word     Weight
        worst -11.130061
          bad  -7.981782
        waste  -7.923940
        awful  -7.886028
       boring  -7.449185
     terrible  -6.595026
         poor  -6.245032
         dull  -5.695035
      nothing  -5.589920
     horrible  -5.288875
        worse  -5.197897
       poorly  -5.087179
unfortunately  -4.980271
        fails  -4.684964
     annoying  -4.608557


In [11]:
import joblib
import os

# Create the models directory 
os.makedirs('../models', exist_ok=True)

# Save the trained Logistic Regression model
joblib.dump(model, '../models/logistic_regression_baseline.pkl')

# Save the fitted TF-IDF Vectorizer
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

print("Model and Vectorizer successfully saved to the models/ folder!")

Model and Vectorizer successfully saved to the models/ folder!


In [16]:
def predict_custom_review(text):
    # Clean the input text using your existing function
    cleaned = clean_text(text)
    
    # Transform using the saved vectorizer (do NOT fit)
    vectorized = tfidf.transform([cleaned])
    
    # Predict probability and class
    prediction = model.predict(vectorized)[0]
    probabilities = model.predict_proba(vectorized)[0]
    
    print(f"Review: \"{text}\"")
    print(f"Predicted Sentiment: {prediction.upper()}")
    print(f"Confidence: Neg: {probabilities[0]:.2%}, Pos: {probabilities[1]:.2%}\n")

# Try it out!
predict_custom_review("This movie was an absolute masterpiece, the acting was phenomenal.")
predict_custom_review("I wanted to like it, but the plot was completely boring and flat.")

Review: "This movie was an absolute masterpiece, the acting was phenomenal."
Predicted Sentiment: POSITIVE
Confidence: Neg: 41.46%, Pos: 58.54%

Review: "I wanted to like it, but the plot was completely boring and flat."
Predicted Sentiment: NEGATIVE
Confidence: Neg: 99.68%, Pos: 0.32%

